# core

> Fill in a module description here

In [158]:
# | default_exp audio_video

In [159]:
# | hide
from nbdev.showdoc import *  # type: ignore

In [160]:
# | export
from typing import List, Tuple, cast, List, Optional, Dict, Union, Type
from pathlib import Path
from seewav import visualize
from moviepy.editor import (
    VideoFileClip,
    AudioFileClip,
    CompositeVideoClip,
    TextClip,
    ColorClip,
    concatenate_videoclips,
)
from pydantic import BaseModel, TypeAdapter
import assemblyai as aai
import requests
import os
import numpy as np
from dotenv import load_dotenv
from product_horse.db import Utterance, AbstractDatabase, ValidString, Word
from collections import defaultdict
import tempfile


load_dotenv()

True

In [161]:
# | export
class AssemblyAiWord(BaseModel):
    text: str
    start: int
    end: int
    confidence: float
    speaker: str


word_validator = TypeAdapter(List[AssemblyAiWord])


class AssemblyAiUtterance(BaseModel):
    confidence: float
    end: int
    speaker: str
    start: int
    text: str
    words: List[AssemblyAiWord]


class AssemblyAiTranscript(BaseModel):
    text: str
    utterances: List[AssemblyAiUtterance]


class AudioEditor:
    api_key: str | None = None

    def __init__(
        self,
        api_key: str | None = os.getenv("ASSEMBLY_API_KEY")
        if os.getenv("ASSEMBLY_API_KEY")
        else None,
    ):
        if not api_key:
            raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
        self.api_key = api_key

    def upload_for_transcript(self, file_path: str) -> str:
        """Uploads a file for transcription and returns the file URL."""
        if not file_path.startswith("https://"):
            with open(file_path, "rb") as file:
                audio_file = file.read()
        else:
            audio_file = requests.get(file_path).content
        if not self.api_key:
            raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
        response = requests.post(
            "https://api.assemblyai.com/v2/upload",
            headers={
                "Authorization": self.api_key,
                "Content-Type": "application/octet-stream",
            },
            data=audio_file,
        )
        if response.status_code != 200:
            raise Exception(f"Failed to upload file: {response.text}")
        file_url = response.json()["upload_url"]
        return file_url

    async def transcribe(self, file_url: str) -> AssemblyAiTranscript:
        """Transcribes a file and returns the transcript."""
        aai.settings.api_key = self.api_key
        config = aai.TranscriptionConfig(speaker_labels=True)
        transcript = aai.Transcriber().transcribe(file_url, config)
        text_well_formed: bool = (
            transcript.utterances
            and transcript.text
            and len(transcript.text) > 0
            and len(transcript.utterances) > 0
        )  # type: ignore
        if not text_well_formed:
            raise Exception(f"No text or utterances found in audio file: {file_url}")
        if transcript.status != "error" and text_well_formed:
            # Convert each utterance and its words to the correct Pydantic models
            transcript_text = "\n".join(
                [
                    f"Speaker {utterance.speaker}: {utterance.text}"
                    for utterance in transcript.utterances
                ]
            )
            utterances = [
                AssemblyAiUtterance(
                    confidence=u.confidence,
                    end=u.end,
                    speaker=u.speaker or "speaker not detected",
                    start=u.start,
                    text=u.text,
                    words=word_validator.validate_python(
                        u.words, from_attributes=True
                    ),  # Convert words for each utterance
                )
                for u in transcript.utterances
            ]
            return AssemblyAiTranscript(text=transcript_text, utterances=utterances)
        else:
            raise Exception(f"{transcript.error} - {transcript.status}")

    def make_webvtt(self, utterances: List[AssemblyAiUtterance]):
        # https://developer.mozilla.org/en-US/docs/Web/API/WebVTT_API
        # add later
        pass

In [162]:
# | export
def make_mp4_animation(
    audio_path: str,
    output_path: str,
    seek: int = 0,  # start at the beginning
    duration: int = 10,  # duration in seconds
    rate: int = 30,  # frames per second
    bars: int = 60,  # number of bars in the visualization
    speed: int = 5,  # speed of transitions
    time: float = 0.5,  # amount of audio shown at once on a frame
    oversample: int = 5,  # frequency of changes
    fg_color: Tuple[float, float, float] = (0.2, 0.5, 0.8),  # foreground color
    bg_color: Tuple[float, float, float] = (1, 1, 1),  # background color
    size: Tuple[int, int] = (640, 480),  # size of the output video
) -> None:
    """
    Generate the visualisation for the `audio` file, using a `tmp` folder and saving the final
    video in `out`.
    `seek` and `durations` gives the extract location if any.
    `rate` is the framerate of the output video.

    `bars` is the number of bars in the animation.
    `speed` is the base speed of transition. Depending on volume, actual speed will vary
        between 0.5 and 2 times it.
    `time` amount of audio shown at once on a frame.
    `oversample` higher values will lead to more frequent changes.
    `fg_color` is the rgb color to use for the foreground.
    `bg_color` is the rgb color to use for the background.
    `size` is the `(width, height)` in pixels to generate.
    """
    with tempfile.TemporaryDirectory() as tmp_directory:
        visualize(
            audio=Path(audio_path),
            tmp=Path(tmp_directory),
            out=Path(output_path),
            seek=seek,
            duration=duration,
            rate=rate,
            bars=bars,
            speed=speed,
            time=time,
            oversample=oversample,
            fg_color=fg_color,
            bg_color=bg_color,
            size=size,
        )

In [163]:
# | export
from collections import OrderedDict

ClipType = Union[VideoFileClip, AudioFileClip, CompositeVideoClip]


def _group_utterances_by_transcript_id(
    utterances: List[Utterance],
) -> OrderedDict[str, List[Utterance]]:
    """Groups utterances by transcript_id using defaultdict."""
    transcript_utterances_by_id: OrderedDict[str, List[Utterance]] = OrderedDict()
    for utterance in utterances:
        transcript_id = str(utterance.transcription_id)
        if transcript_id not in transcript_utterances_by_id:
            transcript_utterances_by_id[transcript_id] = []
        transcript_utterances_by_id[transcript_id].append(utterance)
    return transcript_utterances_by_id


def _get_video_properties(file_path: str):
    clip = VideoFileClip(file_path)
    frame_rate: int = cast(int, clip.fps)
    resolution: Tuple[int, int] = cast(Tuple[int, int], clip.size)
    duration: float = clip.duration
    aspect_ratio: float = clip.aspect_ratio
    clip.close()
    return frame_rate, resolution, duration, aspect_ratio


def make_word_clips_from_file_path_and_words(
    file_path: str,
    words: List[Word],
    target_resolution: Tuple[int, int],
    start: float,
    end: float = 0.0,
    position: str = "center",
) -> CompositeVideoClip:
    """word_start is indexed against the start of the transcript, not necessarily the video clip. So the start of the video clip is used as an offset."""
    clips_buffer = []
    word_clips = []
    previous_word_end = 0.0
    gap_between_words = 0.0
    # start of this video_clip should be used as 0
    if end > 0:
        video_clip = VideoFileClip(
            file_path, target_resolution=target_resolution
        ).subclip(start, end)
    else:
        video_clip = VideoFileClip(file_path, target_resolution=target_resolution)
    text_positions = {
    "center": ("center","center"),
    "bottom-third": ("center","bottom")
        }
    text_position = text_positions.get(position, (0.5, 0.5))
    print('text_position', text_position)
    for word in words:
        word_start = float(word.start) / 1000
        word_start_adj = word_start - start
        word_end = float(word.end) / 1000
        word_end_adj = word_end - start
        # print('word_end_adj', word_end_adj)
        # print('word_start_adj', word_start_adj)
        # print('word_end', word_end)
        # print('word_start', word_start)
        final_end = video_clip.duration
        if word_start_adj > final_end:
            print("word_start_adj is greater than final_end")
            final_word_clip = concatenate_videoclips(word_clips).set_position(text_position)
            final_clip = CompositeVideoClip([video_clip, final_word_clip])
            return final_clip
        if word_end_adj > final_end:
            print("word_end_adj is greater than final_end")
            word_end_adj = final_end
        word_text = word.text
        word_color = (
            "white"
            if word.speaker == "A"
            else "yellow"
            if word.speaker == "B"
            else "grey"
        )
        word_clip = TextClip(
            word_text,
            fontsize=48,
            size=(target_resolution[1], target_resolution[0]/3), #because of the weird resolution flipping issue
            color=word_color,
            stroke_color="black",
            font="Helvetica",
            stroke_width=2,
        )
        duration = word_end_adj - word_start_adj
        word_clip = word_clip.set_duration(
            duration + gap_between_words # gap between words seems to be pushing a little too far ahead vs audio, revisit
        ) # setting position here doesn't seem to work at all
        word_clips.append(word_clip)
        gap_between_words = word_start_adj - previous_word_end
        previous_word_end = word_end_adj
    final_word_clip = concatenate_videoclips(word_clips).set_position(text_position)
    final_clip = CompositeVideoClip([video_clip, final_word_clip])
    return final_clip


def _get_audio_properties(file_path: str):
    clip = AudioFileClip(file_path)
    clip.close()
    return clip.fps, None, clip.duration


def make_video_from_utterances(
    utterances: List[Utterance], database: AbstractDatabase, video_title="Product Horse"
):
    """List of utterances to video. Utterances will be processed in the order they are recieved."""
    transcript_utterances_by_id = _group_utterances_by_transcript_id(utterances)
    list_of_transcript_ids = [
        str(transcript_id) for transcript_id in transcript_utterances_by_id.keys()
    ]
    metadata = database.get_file_metadata_from_list_of_transcript_ids(
        list_of_transcript_ids
    )
    if len(metadata) == 0:
        raise ValueError(
            f"No metadata found, important variables listed: {list_of_transcript_ids}"
        )
    transcript_utterances_by_id_with_file_path: Dict[
        str, Dict[str, Union[ValidString, List[Utterance], bool]]
    ] = {}
    resolutions: List[Tuple[int, int]] = []
    frame_rates: List[int] = []
    frame_rate: Optional[int] = 30
    resolution: Optional[Tuple[int, int]] = None
    aspect_ratios: List[float] = []
    reordered_metadata = []
    audio_files_to_process = []
    for transcript_id in transcript_utterances_by_id.keys():
        for meta in metadata:
            if str(meta.transcription.id) == transcript_id:
                reordered_metadata.append(meta)
                break
    for meta in reordered_metadata:
        video_flag = meta.file_path.endswith(".mp4")
        if video_flag:
            frame_rate, resolution, duration, aspect_ratio = _get_video_properties(
                meta.file_path
            )
        else:
            frame_rate, resolution, duration = _get_audio_properties(meta.file_path)
        if frame_rate is None:
            raise ValueError(
                f"No frame rate or resolution found for {video_flag and 'video' or 'audio'} file {meta.file_path}"
            )
        if resolution is not None:
            resolutions.append(resolution)
            frame_rates.append(frame_rate)
            aspect_ratios.append(aspect_ratio)
        utterance_by_id = transcript_utterances_by_id[str(meta.transcription.id)]
        transcript_utterances_by_id_with_file_path[str(meta.transcription.id)] = {
            "file_path": meta.file_path,
            "utterances": utterance_by_id,
            "video": video_flag,
            "user_name": database.get_user(meta.user_id).name,
        }
    if len(transcript_utterances_by_id_with_file_path) == 0:
        raise ValueError("No transcripts found")
    max_resolution: Tuple[int, int] = (1280, 720)
    max_fps: int = 30
    if len(resolutions) > 0:
        max_resolution = np.amax(resolutions, axis=0)
    if len(frame_rates) > 0:
        max_fps = np.max(frame_rates)
    transcript_videos: List[ClipType] = []
    # at some point the target resolution is reversed internally in moviepy for videoclips, so reverse it here to counteract that.
    target_resolution = (int(max_resolution[1]), int(max_resolution[0]))
    target_resolution_normal = (int(max_resolution[0]), int(max_resolution[1]))
    with tempfile.TemporaryDirectory() as tmp_directory:
        os.mkdir(tmp_directory + "/audio")
        os.mkdir(tmp_directory + "/visualizations")
        for _, data in transcript_utterances_by_id_with_file_path.items():
            file_path: str = data["file_path"]
            user_name: str = data["user_name"]
            if os.path.exists(file_path) is False:
                raise ValueError(f"File not found: {file_path}")
            clip_utterances: List[Utterance] = data["utterances"]
            video_flag: bool = data["video"]
            clips_buffer: List[ClipType] = []
            for clip_utterance in clip_utterances:
                clip_words = database.get_words_from_utterance_ids([clip_utterance.id])
                start: float = float(clip_utterance.start) / 1000
                end: float = float(clip_utterance.end) / 1000
                clip_duration = int(end - start)
                if not video_flag:
                    """handle the audio file conversion to video"""
                    working_clip: ClipType = AudioFileClip(file_path).subclip(
                        start, end
                    )  # type: ignore
                    temp_audio_path = (
                        f"{tmp_directory}/audio/{transcript_id}_{clip_utterance.id}.mp3"
                    )
                    working_clip.write_audiofile(temp_audio_path)
                    working_clip.close()
                    # Visualize the audio segment
                    output_viz_path = f"{tmp_directory}/visualizations/{transcript_id}_{clip_utterance.id}.mp4"
                    if (
                        clip_duration > 1
                    ):  # if you remove this you will get failures with exit status 254 from ffmeg
                        make_mp4_animation(
                            audio_path=temp_audio_path,
                            output_path=output_viz_path,
                            duration=clip_duration,
                            rate=max_fps,
                            size=target_resolution_normal,
                        )
                        word_clip = make_word_clips_from_file_path_and_words(
                            output_viz_path,
                            clip_words,
                            target_resolution,
                            start,
                            position="center",
                        )
                        clips_buffer.append(word_clip)
                else:
                    word_clip = make_word_clips_from_file_path_and_words(
                        file_path,
                        clip_words,
                        target_resolution,
                        start,
                        end,
                        position="bottom-third",
                    )
                    clips_buffer.append(word_clip)
                    # video_clip = VideoFileClip(
                    #     file_path, target_resolution=target_resolution
                    # ).subclip(start, end)
                    # clips_buffer.append(video_clip)
                # TODO: Adjust playback speed to match the new frame rate
                # Potential implementation:
                # if clip.fps != max_fps:
                #     speed_factor = clip.fps / max_fps
                #     clip = clip.fx(vfx.speedx, speed_factor)
                #     clip = clip.set_fps(max_fps)
            for clip in clips_buffer:
                if not hasattr(clip, "fps") or clip.fps is None:
                    clip.fps = max_fps
            transcript_final_clip = concatenate_videoclips(clips=clips_buffer)
            print("transcript clip size", transcript_final_clip.size)
            if "!" in user_name:
                transcript_videos.append(transcript_final_clip)
            else:
                metadata_banner = TextClip(
                    user_name,
                    fontsize=48,
                    color="white",
                    stroke_color="black",
                    font="Helvetica",
                    stroke_width=2,
                )
                metadata_banner = metadata_banner.set_position(
                    (0.05, 0.05),
                    relative=True,
                ).set_duration(transcript_final_clip.duration)
                final_clip = CompositeVideoClip(
                    [transcript_final_clip, metadata_banner]
                )
                final_clip.fps = max_fps
                transcript_videos.append(final_clip)
        video_title_safe = (
            video_title.replace(" ", "_")
            .replace("'", "")
            .replace("?", "")
            .replace(":", "")
            .replace("$", "")
            .replace(".", "")
            .replace("/", "")
            .replace("\\", "")
        )
        if len(video_title_safe) > 50:
            video_title_safe = video_title_safe[:50]
        file_path = f"../temp-videos/{video_title_safe}.mp4"
        final_video = concatenate_videoclips(transcript_videos)
        with final_video as video:
            video.write_videofile(
                file_path,
                fps=max_fps,
                audio_codec="aac",
                preset="ultrafast",
                codec="libx264",
                threads=os.cpu_count(),
            )
    return file_path

In [164]:
# Weird issue with resolution being off I think
# add transcript to stream

In [165]:
from product_horse.db import SqlModelDatabase

PG_URL = "postgresql://localhost:5432/product_horse"
db = SqlModelDatabase(database_url=PG_URL)
users = db.get_all_users()
transcripts = db.get_all_unique_transcriptions(mode="file_name")
test_transcript = transcripts[2]
test_transcript_2 = transcripts[5]
test_transcript_3 = transcripts[10]
test_transcript_4 = transcripts[11]
test_transcript_5 = transcripts[20]
test_utterances = (
    # test_transcript.utterances[10:20]
    # + test_transcript_2.utterances[10:20]
    # + test_transcript_3.utterances[10:20]
    test_transcript_5.utterances[10:11]
    + test_transcript_4.utterances[10:11]
)
make_video_from_utterances(test_utterances, db)

MoviePy - Writing audio in /var/folders/qj/d65y3xj97dj5gmv5gpb9bq3m0000gn/T/tmpt5ms40ni/audio/e53adc6f-89e2-4549-a8f4-b00b5576e615_84810d55-5942-4502-9993-07a77db2d1cc.mp3


MoviePy - Done.


Generating the frames...


100%|████████████████████████████████████| 270/270 [00:06<00:00, 42.78 frames/s]


Encoding the animation video... 
text_position ('center', 'center')
word_end_adj is greater than final_end
word_start_adj is greater than final_end
transcript clip size (1280, 720)
text_position ('center', 'bottom')
transcript clip size (1280, 720)
Moviepy - Building video ../temp-videos/Product_Horse.mp4.
MoviePy - Writing audio in Product_HorseTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video ../temp-videos/Product_Horse.mp4



Moviepy - Done !
Moviepy - video ready ../temp-videos/Product_Horse.mp4


'../temp-videos/Product_Horse.mp4'

In [166]:
FILE_URL = "https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3"

In [167]:
make_mp4_animation("../temp-videos/wildfires.mp3", "../temp-videos/test-viz.mp4")

Generating the frames...


100%|███████████████████████████████████| 300/300 [00:02<00:00, 117.46 frames/s]


Encoding the animation video... 


In [168]:
def test_upload_for_transcript():
    api_key = os.getenv("ASSEMBLYAI_API_KEY")
    if not api_key:
        raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
    audio_editor = AudioEditor(api_key=api_key)
    for url in [FILE_URL, "../bleat.m4a"]:
        file_url = audio_editor.upload_for_transcript(url)
        assert file_url is not None, "File URL is None"
        assert isinstance(file_url, str), "File URL is not a string"


test_upload_for_transcript()

In [169]:
EXPECTED_TEXT = """Speaker A: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US. Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, the air quality warnings include the warning to stay inside. We wanted to better understand what's happening here and why, so we called Peter DiCarlo, an associate professor in the department of Environmental Health and Engineering at Johns Hopkins University. Good morning. Professor Good morning. So what is it about the conditions right now that have caused this round of wildfires to affect so many people so far away?
Speaker B: Well, there's a couple of things. The season has been pretty dry already, and then the fact that we're getting hit in the US is because there's a couple of weather systems that are essentially channeling the smoke from those Canadian wildfires through Pennsylvania into the mid Atlantic and the northeast and kind of just dropping the smoke there.
Speaker A: So what is it in this haze that makes it harmful? And I'm assuming it is.
Speaker B: Is it is. The levels outside right now in Baltimore are considered unhealthy. And most of that is due to what's called particulate matter, which are tiny particles, microscopic, smaller than the width of your hair, that can get into your lungs and impact your respiratory system, your cardiovascular system, and even your neurological, your brain.
Speaker A: What makes this particularly harmful? Is it the volume of particulate? Is it something in particular? What is it exactly? Can you just drill down on that a little bit more? Yeah.
Speaker B: So the concentration of particulate matter I was looking at, some of the monitors that we have was reaching levels of what are, in science speak, 150 micrograms per meter cubed, which is more than ten times what the annual average should be and about four times higher than what you're supposed to have on a 24 hours average. And so the concentrations of these particles in the air are just much, much higher than we typically see. And exposure to those high levels can lead to a host of health problems.
Speaker A: And who is most vulnerable? I noticed that in New York City, for example, they're canceling outdoor activities. And so here it is in the early days of summer, and they have to keep all the kids inside. So who tends to be vulnerable in a situation like this?
Speaker B: It's the youngest. So children, obviously, whose bodies are still developing, the elderly who know their bodies are more in decline and they're more susceptible to the health impacts of breathing, the poor air quality. And then people who have preexisting health conditions, people with respiratory conditions or heart conditions, can be triggered by high levels of air pollution.
Speaker A: Could this get worse?
Speaker B: That's a good, in some areas it's much worse than others, and it just depends on kind of where the smoke is concentrated. I think New York has some of the higher concentrations right now, but that's going to change as that air moves away from the New York area. But over the course of the next few days, we will see different areas being hit at different times with the highest concentrations. I was going to ask you more fires start burning. I don't expect the concentrations to go up too much higher.
Speaker A: I was going to ask you, and you started to answer this, but how much longer could this last? Or, forgive me if I'm asking you to speculate, but what do you think?
Speaker B: Well, I think the fires are going to burn for a little bit longer, but the key for us in the US is the weather system changing. And so right now it's kind of the weather systems that are pulling that air into our mid atlantic and northeast region. As those weather systems change and shift, we'll see that smoke going elsewhere and not impact us in this region as much. And so I think that's going to be the defining factor. And I think the next couple of days we're going to see a shift in that weather pattern and start to push the smoke away from where we are.
Speaker A: And finally, with the impacts of climate change, we are seeing more wildfires. Will we be seeing more of these kinds of wide ranging air quality consequences or circumstances?
Speaker B: I mean, that is one of the predictions for climate change. Looking into the future, the fire season is starting earlier and lasting longer, and we're seeing more frequent fires. So, yeah, this is probably something that we'll be seeing more frequently. This tends to be much more of an issue in the western Us. So the eastern us getting hit right now is a little bit new. But, yeah, I think with climate change moving forward, this is something that is going to happen more frequently.
Speaker A: That's Peter DiCarlo, associate professor in the department of Environmental Health and engineering at Johns Hopkins University. Thanks so much for joining us and sharing this expertise with us.
Speaker B: Thank you for having me."""
EXPECTED_UTTERANCES_COUNT = 16


async def test_transcribe():
    api_key = os.getenv("ASSEMBLYAI_API_KEY")
    if not api_key:
        raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
    audio_editor = AudioEditor(api_key=api_key)
    transcript = await audio_editor.transcribe(FILE_URL)
    print(transcript)
    print(len(transcript.utterances))
    assert "New York" in transcript.text, "Transcript text does not match expected text"
    assert (
        len(transcript.utterances) >= EXPECTED_UTTERANCES_COUNT
    ), "Number of utterances does not match expected count"
    # check words exist
    assert all(
        utterance.words for utterance in transcript.utterances
    ), "Some utterances do not have words"


await test_transcribe()

text="Speaker A: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US. Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, the air quality warnings include the warning to stay inside. We wanted to better understand what's happening here and why. So he called Peter DiCarlo, an associate professor in the department of Environmental Health and Engineering at Johns Hopkins University. Good morning. Professor.\nSpeaker B: Good morning.\nSpeaker A: So what is it about the conditions right now that have caused this round of wildfires to affect so many people so far away?\nSpeaker B: Well, there's a couple of things. The season has been pretty dry already, and then the fact that we're getting hit in the US is because there's a couple weather systems that are essentially channeling the smoke from those canadian wildfires through Pennsylvania into the mid Atlantic in the northeast and kind of just dropping the smoke there

In [170]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore